In [19]:
import pandas as pd
import glob
from tqdm import tqdm
import requests
import time
import os
import shutil
import matplotlib.pyplot as plt
import sys
import platform

def load_manifest_file(location):
    """
    Function use:
    Load the GDC manifest file from a specified directory.

    Inputs:
    - location (str): Directory path containing the GDC manifest file.

    Outputs:
    - DataFrame: Contains the data from the manifest file if found.
    - None: If no matching file is found.
    """
    pattern = f"{location}/*gdc_manifest*"
    files = glob.glob(pattern)
    if files:
        return pd.read_csv(files[0], sep='\t')
    else:
        return None

def fetch_sample_type_from_file_uuid(file_uuid):
    """
    Function use:
    Fetch the sample type for a given file UUID using the GDC API.

    Inputs:
    - file_uuid (str): UUID of the file.

    Outputs:
    - sample_type (str): Sample type (e.g., 'Primary Tumor').
    - None: If the API request fails.
    """
    base_url = 'https://api.gdc.cancer.gov'
    response = requests.get(f'{base_url}/files/{file_uuid}?expand=cases.samples')
    if response.status_code == 200:
        data = response.json()
        return data.get('data', {}).get('cases', [{}])[0].get('samples', [{}])[0].get('sample_type', 'Not Found')
    else:
        print(f"Failed to retrieve sample type for UUID {file_uuid}.")
        return None

def fetch_demographic_details_from_file_uuid(file_uuid):
    """
    Function use:
    Fetch demographic details for a given file UUID using the GDC API.

    Inputs:
    - file_uuid (str): UUID of the file.

    Outputs:
    - demographic_details (dict): Dictionary containing demographic information.
    - None: If the API request fails or no data is found.
    """
    base_url = 'https://api.gdc.cancer.gov'
    response = requests.get(f'{base_url}/files/{file_uuid}?expand=cases.demographic')
    if response.status_code == 200:
        data = response.json()
        return data.get('data', {}).get('cases', [{}])[0].get('demographic', {})
    else:
        print(f"Failed to retrieve demographic details for UUID {file_uuid}.")
        return None

def fetch_diagnosis_details_from_file_uuid(file_uuid):
    """
    Function use:
    Fetch diagnosis details for a given file UUID using the GDC API.

    Inputs:
    - file_uuid (str): UUID of the file.

    Outputs:
    - diagnosis_details (list): List of dictionaries containing diagnosis information.
    - Empty list: If the API request fails.
    """
    base_url = 'https://api.gdc.cancer.gov'
    response = requests.get(f'{base_url}/files/{file_uuid}?expand=cases.diagnoses')
    if response.status_code == 200:
        data = response.json()
        return data.get('data', {}).get('cases', [{}])[0].get('diagnoses', [])
    else:
        print(f"Failed to retrieve diagnosis details for UUID {file_uuid}.")
        return []

# def fetch_clinical_data_for_manifest(manifest, n=None, t=1):
#     """
#     Function use:
#     Enrich a manifest DataFrame with sample types and diagnosis details fetched from the GDC API.

#     Inputs:
#     - manifest (DataFrame): Manifest with file UUIDs in an 'id' column.
#     - n (int): Maximum number of files to process. If None, all rows are processed.
#     - t (float): Time (in seconds) to wait between API calls.

#     Outputs:
#     - Updated DataFrame: Includes additional sample type and diagnosis details.
#     """
#     if 'sample_type' not in manifest.columns:
#         manifest['sample_type'] = None

#     if n is None:
#         n = len(manifest)

#     for i in tqdm(range(min(n, len(manifest)))):
#         if i > 0:
#             time.sleep(t)
#         file_id = manifest.loc[i, 'id']
#         sample_type = fetch_sample_type_from_file_uuid(file_id)
#         if sample_type:
#             manifest.at[i, 'sample_type'] = sample_type
#         else:
#             print(f"No sample type found for file UUID {file_id}.")
#         diagnosis_details = fetch_diagnosis_details_from_file_uuid(file_id)
#         if diagnosis_details and isinstance(diagnosis_details, list):
#             for detail in diagnosis_details:
#                 if isinstance(detail, dict):
#                     for key, value in detail.items():
#                         column_name = f"diagnosis_{key}"
#                         if column_name not in manifest.columns:
#                             manifest[column_name] = None
#                         manifest.at[i, column_name] = value
#                 else:
#                     print(f"Detail is not a dictionary: {detail}")
#         else:
#             print(f"Diagnosis details structure unexpected or missing for file ID {file_id}: {diagnosis_details}")

#     return manifest

def fetch_clinical_data_for_manifest(manifest, n=None, t=1):
    """
    Enrich a manifest DataFrame with sample types, diagnosis details, and demographic data fetched from the GDC API.
    """
    if 'sample_type' not in manifest.columns:
        manifest['sample_type'] = None

    if n is None:
        n = len(manifest)

    for i in tqdm(range(min(n, len(manifest)))):
        if i > 0:
            time.sleep(t)
        file_id = manifest.loc[i, 'id']

        # Fetch and append sample type
        sample_type = fetch_sample_type_from_file_uuid(file_id)
        if sample_type:
            manifest.at[i, 'sample_type'] = sample_type

        # Fetch and append diagnosis details
        diagnosis_details = fetch_diagnosis_details_from_file_uuid(file_id)
        if diagnosis_details and isinstance(diagnosis_details, list):
            for detail in diagnosis_details:
                if isinstance(detail, dict):
                    for key, value in detail.items():
                        column_name = f"diagnosis_{key}"
                        if column_name not in manifest.columns:
                            manifest[column_name] = None
                        manifest.at[i, column_name] = value

        # Fetch and append demographic details
        demographic_details = fetch_demographic_details_from_file_uuid(file_id)
        if demographic_details and isinstance(demographic_details, dict):
            for key, value in demographic_details.items():
                column_name = f"demographic_{key}"
                if column_name not in manifest.columns:
                    manifest[column_name] = None
                manifest.at[i, column_name] = value

    return manifest


def save_clinical_manifest(manifest, directory):
    """
    Function use:
    Save the clinical manifest DataFrame to a file.

    Inputs:
    - manifest (DataFrame): Data to save.
    - directory (str): Directory path where the file will be saved.

    Outputs:
    - None: File is saved directly to the specified directory.
    """
    directory = os.path.join(directory, '')
    filepath = f"{directory}clinical_gdc_manifest.txt"
    manifest.to_csv(filepath, sep='\t', index=False)
    print(f"File saved successfully at {filepath}")

def load_clinical_manifest(directory):
    """
    Function use:
    Load a clinical manifest file into a DataFrame.

    Inputs:
    - directory (str): Directory path where the file is located.

    Outputs:
    - DataFrame: Contains the clinical manifest data.
    """
    filepath = os.path.join(directory, 'clinical_gdc_manifest.txt')
    clinical_manifest = pd.read_csv(filepath, sep='\t')
    print('Clinical manifest shape:', clinical_manifest.shape)
    return clinical_manifest

def clean_dataframe(df):
    """
    Function use:
    Clean a DataFrame by removing columns with irrelevant or missing data.

    Inputs:
    - df (DataFrame): DataFrame to clean.

    Outputs:
    - DataFrame: Cleaned version of the input DataFrame.
    """
    df_copy = df.copy()
    for column in df_copy.columns:
        unique_values = df_copy[column].dropna().unique()
        if len(unique_values) == 0 or (len(unique_values) == 1 and unique_values[0] in ['Not Reported', 'not reported', 'None']):
            df_copy.drop(column, axis=1, inplace=True)
        else:
            if column in ["ajcc_pathologic_stage", "tumor_grade", "ajcc_pathologic_t", "ajcc_pathologic_m",
                          "ajcc_pathologic_n", "figo_stage", "wilms_tumor_histologic_subtype"]:
                df_copy.loc[df_copy['sample_type'] == 'Solid Tissue Normal', column] = 'healthy tissue'
    print('Cleaned clinical manifest shape:', df_copy.shape)
    return df_copy

def organize_tsv_files(base_dir):
    """
    Function use:
    Organize .tsv files by moving them from subdirectories to the base directory.

    Inputs:
    - base_dir (str): Path to the base directory.

    Outputs:
    - None: Files are moved, and subdirectories are cleaned.
    """
    has_directories = False
    for root, dirs, files in os.walk(base_dir, topdown=False):
        if dirs:
            has_directories = True
        for file in files:
            if file.endswith(".tsv"):
                shutil.move(os.path.join(root, file), os.path.join(base_dir, file))
    if has_directories:
        for root, dirs, files in os.walk(base_dir, topdown=False):
            for dir in dirs:
                shutil.rmtree(os.path.join(root, dir))
        print("Cleaning done")
    else:
        print("No directories to clean. Cleaning done.")

def load_gene_expression(manifest_df, exp_dir, feature):
    """
    Function use:
    Load gene expression data from .tsv files, keeping only relevant rows and columns.

    Inputs:
    - manifest_df (DataFrame): Manifest with filenames.
    - exp_dir (str): Directory containing .tsv files.
    - feature (str): Gene expression feature to extract.

    Outputs:
    - gene_expression_df (DataFrame): Contains gene expression data.
    - gene_key_df (DataFrame): Maps gene IDs to gene names.
    """
    gene_expression_data = {}
    gene_key = {}

    for filename in manifest_df['filename']:
        file_path = os.path.join(exp_dir, filename)
        try:
            df = pd.read_csv(file_path, sep='\t', comment='#')
            df = df[df['gene_id'].str.contains('ENSG')]
            gene_expression_data[filename] = df[feature].values
            gene_key.update(dict(zip(df['gene_id'], df['gene_name'])))
        except Exception as e:
            print(f"Failed to load {filename}: {e}")

    gene_expression_df = pd.DataFrame(gene_expression_data, index=list(gene_key.keys()))
    gene_expression_df.index.name = 'gene_id'
    gene_expression_df.columns.name = 'filename'

    gene_key_df = pd.DataFrame(list(gene_key.items()), columns=['gene_id', 'gene_name'])

    print('Gene expression shape:', gene_expression_df.T.shape)
    print('Gene key shape:', gene_key_df.shape)

    return gene_expression_df.T, gene_key_df

def load_gene_expression_with_filter(manifest_df, exp_dir, feature, gene_list):
    """
    Load gene expression data from .tsv files, filtering by a specific list of genes.

    Inputs:
    - manifest_df (DataFrame): Manifest with filenames.
    - exp_dir (str): Directory containing .tsv files.
    - feature (str): Gene expression feature to extract.
    - gene_list (list): List of gene names to filter by.

    Outputs:
    - filtered_gene_expression_df (DataFrame): Contains gene expression data for selected genes.
    """
    gene_expression_data = {}
    gene_key = {}

    for filename in manifest_df['filename']:
        file_path = os.path.join(exp_dir, filename)
        try:
            # Load gene expression data
            df = pd.read_csv(file_path, sep='\t', comment='#')

            # Filter for ENSG IDs and extract relevant rows
            df = df[df['gene_id'].str.contains('ENSG')]

            # Map gene IDs to gene names
            gene_key.update(dict(zip(df['gene_id'], df['gene_name'])))

            # Filter for genes in the provided gene list
            filtered_df = df[df['gene_name'].isin(gene_list)]

            # Add expression values for the filtered genes
            gene_expression_data[filename] = filtered_df[feature].values
        except Exception as e:
            print(f"Failed to load {filename}: {e}")

    # Create DataFrame for filtered gene expression data
    filtered_gene_ids = [gene_id for gene_id, gene_name in gene_key.items() if gene_name in gene_list]
    filtered_gene_expression_df = pd.DataFrame(gene_expression_data, index=filtered_gene_ids)

    # Map gene IDs to gene names
    filtered_gene_expression_df.index = [gene_key[gene_id] for gene_id in filtered_gene_expression_df.index]
    filtered_gene_expression_df.index.name = 'gene_name'
    filtered_gene_expression_df.columns.name = 'filename'

    return filtered_gene_expression_df

# Save updated gene expression data with selected genes
def save_filtered_gene_expression(filtered_gene_exps, cancer):
    """
    Save filtered gene expression data to a CSV file.

    Inputs:
    - filtered_gene_exps (DataFrame): Filtered gene expression data.
    - cancer (str): Directory for saving the data.

    Outputs:
    - None: DataFrame is saved to a file.
    """
    filtered_gene_exps.to_csv(f'./data/TCGA_GeneExpression/{cancer}/filtered_gene_expression.csv', index=True)

# Example usage
gene_list = ["OGT", "OGA"]  # Input gene list
filtered_gene_exps = load_gene_expression_with_filter(
    manifest_df=clinical_manifest,  # Provide the manifest DataFrame
    exp_dir=f'./data/TCGA_GeneExpression/{cancer}/gene_expression',  # Directory with .tsv files
    feature='fpkm_unstranded',  # Specify the feature column
    gene_list=gene_list  # Specify the gene list to filter
)
save_filtered_gene_expression(filtered_gene_exps, cancer)

# Check the output
print(filtered_gene_exps.head())


def save_dataframes(gene_exps, gene_key, cancer):
    """
    Function use:
    Save gene expression and key DataFrames to files.

    Inputs:
    - gene_exps (DataFrame): Gene expression data.
    - gene_key (DataFrame): Gene key data.
    - cancer (str): Directory for saving the data.

    Outputs:
    - None: DataFrames are saved to files.
    """
    gene_exps.to_csv(f'./data/TCGA_GeneExpression/{cancer}/gene_expression.csv', index=True)
    gene_key.to_csv(f'./data/TCGA_GeneExpression/{cancer}/gene_key.csv', index=True)

def load_dataframes(exps_filename, key_filename):
    """
    Function use:
    Load gene expression and key DataFrames from files.

    Inputs:
    - exps_filename (str): Path to gene expression file.
    - key_filename (str): Path to gene key file.

    Outputs:
    - gene_exps (DataFrame): Loaded gene expression data.
    - gene_key (DataFrame): Loaded gene key data.
    """
    gene_exps = pd.read_csv(exps_filename, index_col=0)
    gene_key = pd.read_csv(key_filename, index_col=0)
    return gene_exps, gene_key

def analyze_filenames(manifest_df, directory_path):
    """
    Analyze the uniqueness and overlap of filenames between a manifest and a directory.

    Inputs:
    - manifest_df (DataFrame): Manifest containing filenames.
    - directory_path (str): Directory path containing files to compare.

    Outputs:
    - dict: Summary of percentages including unique filenames, overlap, and non-overlap.
    """
    total_filenames = len(manifest_df['filename'])
    unique_filenames = len(manifest_df['filename'].unique())
    percent_unique = (unique_filenames / total_filenames) * 100

    directory_files = set(os.listdir(directory_path))
    manifest_files = set(manifest_df['filename'])

    overlap_files = manifest_files.intersection(directory_files)
    percent_overlap = (len(overlap_files) / total_filenames) * 100
    non_overlap_files = manifest_files.difference(directory_files)
    percent_non_overlap = (len(non_overlap_files) / total_filenames) * 100

    return {
        "percent_unique_filenames_in_manifest": percent_unique,
        "percent_overlap_with_directory": percent_overlap,
        "percent_non_overlap_with_directory": percent_non_overlap
    }

def create_clinical_expression_survival(filtered_gene_exps, clinical_manifest, cancer, save_dir):
    """
    Create and save a combined DataFrame with gene expression, cancer group, clinical, and demographic data.

    Inputs:
    - filtered_gene_exps (DataFrame): Filtered gene expression data (genes as rows, samples as columns).
    - clinical_manifest (DataFrame): Clinical manifest with clinical and demographic information.
    - cancer (str): Cancer group name (e.g., "uterus").
    - save_dir (str): Directory to save the resulting CSV file.

    Outputs:
    - combined_df (DataFrame): Combined DataFrame with all the information.
    """
    # Transpose gene expression data for merging (samples as rows, genes as columns)
    filtered_gene_exps = filtered_gene_exps.T.reset_index()
    filtered_gene_exps.rename(columns={'index': 'filename'}, inplace=True)

    # Add a 'cancer_group' column
    filtered_gene_exps['cancer_group'] = cancer

    # Merge clinical manifest with filtered gene expression data
    combined_df = pd.merge(
        filtered_gene_exps,  # Gene expression data
        clinical_manifest,  # Clinical and demographic data
        how='inner',  # Only keep rows where filenames match
        left_on='filename',
        right_on='filename'
    )

    # Save the resulting DataFrame
    output_path = os.path.join(save_dir, "clinical_expression_survival.csv")
    combined_df.to_csv(output_path, index=False)
    print(f"Combined DataFrame saved at: {output_path}")

    return combined_df

# Show Python version, platform details, and versions of all imported packages.
def environment_info():
    """
    Display Python version, platform details, and versions of all imported packages.
    """
    # Python and system information
    print("Python version:", sys.version)
    print("Platform:", platform.platform())

    # Imported package versions
    print("\nPackage Versions:")
    imported_modules = {name: module.__version__ for name, module in sys.modules.items() 
                        if hasattr(module, '__version__')}
    for package, version in sorted(imported_modules.items()):
        print(f"{package}: {version}")


filename   6a6ccdf5-9405-409a-a161-72e875400889.rna_seq.augmented_star_gene_counts.tsv  \
gene_name                                                                                
OGT                                                  82.2884                             
OGA                                                  66.1513                             

filename   e5325dd8-867a-4cc2-8b93-9b0b58fc32f3.rna_seq.augmented_star_gene_counts.tsv  \
gene_name                                                                                
OGT                                                  60.7598                             
OGA                                                  58.0955                             

filename   75073929-f7b1-4165-a786-60f733b962d9.rna_seq.augmented_star_gene_counts.tsv  \
gene_name                                                                                
OGT                                                  41.4528                             
OGA     

In [4]:
# Display environment detail
environment_info()


Python version: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 21:00:12) [Clang 16.0.6 ]
Platform: macOS-14.5-x86_64-i386-64bit

Package Versions:
IPython: 8.25.0
IPython.core.release: 8.25.0
PIL: 10.3.0
PIL.Image: 10.3.0
PIL._deprecate: 10.3.0
PIL._version: 10.3.0
_brotli: 1.0.9
_csv: 1.0
_ctypes: 1.1.0
_curses: b'2.2'
_decimal: 1.70
_pydev_bundle.fsnotify: 0.1.5
_pydevd_frame_eval.vendored.bytecode: 0.13.0.dev
appnope: 0.1.4
argparse: 1.1
brotli: 1.0.9
certifi: 2024.02.02
cffi: 1.16.0
charset_normalizer: 2.0.4
charset_normalizer.version: 2.0.4
comm: 0.2.2
csv: 1.0
ctypes: 1.1.0
ctypes.macholib: 1.0
cycler: 0.12.1
dateutil: 2.9.0
dateutil._version: 2.9.0
debugpy: 1.6.7
debugpy.public_api: 1.6.7
decimal: 1.70
decorator: 5.1.1
defusedxml: 0.7.1
executing: 2.0.1
executing.version: 2.0.1
http.server: 0.6
idna: 3.4
idna.idnadata: 15.0.0
idna.package_data: 3.4
ipaddress: 1.0
ipykernel: 6.29.3
ipykernel._version: 6.29.3
jedi: 0.19.1
json: 2.0.9
jupyter_client: 8.6.2
jupyter_client._v

In [28]:
# Specify the cancer type to analyze 
# Analyzed in manuscript ("uterus", "thyroid", "lung", "kidney", "breast", "bone_marrow_and_blood")
cancer = "uterus"
gene_list = ["HIF1A", "OGT"]

# --- Prerequisites ---
# Before running this script, ensure the following:
# 1. A directory structure exists: './data/TCGA_GeneExpression/{cancer}/'
#    (e.g., './data/TCGA_GeneExpression/uterus/').
# 2. Within the above directory, the following must be present:
#    - A GDC manifest txt file with 'gdc_manifest' in its name. This file must be downloaded
#      from the GDC Data Portal and placed in './data/TCGA_GeneExpression/{cancer}/'.
#    - A subdirectory named 'gene_expression' containing all downloaded gene expression `.tsv` files
#      (also from the GDC Data Portal) for the specified cancer type.  
#      (select cancer type filter > transcriptome profiling > Gene Expression Quantification > RNA-Seq > '.tsv' files)


# --- Step 1: Load the manifest file from GDC ---
# This step loads the manifest file listing metadata about available files for the specified cancer type.
gdc_manifest = load_manifest_file(f'./data/TCGA_GeneExpression/{cancer}/')

# --- Step 2: Fetch clinical data using the GDC API ---
# This step queries the GDC API to fetch clinical data such as diagnosis details and sample types.
# Ensure an active internet connection to access the GDC API.
clinical_manifest = fetch_clinical_data_for_manifest(gdc_manifest, n=None, t=0)

# Save the clinical data for reuse. This file will be saved as 'clinical_gdc_manifest.txt'.
save_clinical_manifest(clinical_manifest, f'./data/TCGA_GeneExpression/{cancer}/')

# --- Step 3: Load and clean the clinical manifest ---
# Load the saved clinical manifest file and clean it by removing irrelevant or empty columns.
clinical_manifest = load_clinical_manifest(f'./data/TCGA_GeneExpression/{cancer}/')
clinical_manifest = clean_dataframe(clinical_manifest)

# --- Step 4: Organize and compile gene expression data ---
# Ensure that the subdirectory './data/TCGA_GeneExpression/{cancer}/gene_expression/' exists
# and contains the downloaded gene expression `.tsv` files from the GDC Data Portal.

##### RUN BELOW ONLY IF CLINICAL/SURVIVAL DATA ALREADY RETREIVED! 

# Move all `.tsv` files from subdirectories into the 'gene_expression' directory.
organize_tsv_files(f'./data/TCGA_GeneExpression/{cancer}/gene_expression')

# Load the gene expression data from the `.tsv` files, extracting the specified feature ('fpkm_unstranded').
gene_exps = load_gene_expression_with_filter(
    manifest_df=clinical_manifest,
    exp_dir=f'./data/TCGA_GeneExpression/{cancer}/gene_expression',
    feature='fpkm_unstranded',
    gene_list=gene_list
)

# Save the processed gene expression and gene key data for later use.
save_dataframes(gene_exps, gene_key, cancer)

# --- Step 5: Analyze and verify filenames ---
# Compare filenames in the clinical manifest with those in the 'gene_expression' directory
# to identify any mismatches or missing files.
print(analyze_filenames(clinical_manifest, f'./data/TCGA_GeneExpression/{cancer}/gene_expression'))

# --- Step 6: Reload saved gene expression and gene key data ---
# Reload the saved gene expression and gene key data from the CSV files.
# This is useful for reanalysis or downstream processing without repeating earlier steps.
gene_exps, gene_key = load_dataframes(
    f'./data/TCGA_GeneExpression/{cancer}/gene_expression.csv', 
    f'./data/TCGA_GeneExpression/{cancer}/gene_key.csv'
)

# Create and save the combined DataFrame
clinical_expression_survival_df = create_clinical_expression_survival(
    filtered_gene_exps=gene_exps,
    clinical_manifest=clinical_manifest,
    cancer=cancer,
    save_dir=f'./data/TCGA_GeneExpression/{cancer}/'
)

# Display the first few rows of the new DataFrame
clinical_expression_survival_df.head()



,filename,HIF1A,OGT,cancer_group,id,md5,size,state,sample_type,diagnosis_ajcc_pathologic_stage,...,demographic_submitter_id,demographic_days_to_birth,demographic_created_datetime,demographic_year_of_birth,demographic_demographic_id,demographic_updated_datetime,demographic_age_is_obfuscated,demographic_days_to_death,demographic_state,demographic_year_of_death
0,6a6ccdf5-9405-409a-a161-72e875400889.rna_seq.a...,85.2968,82.2884,uterus,dbf101e0-64b2-493a-8637-e0d465264f3b,87e662c98bd5be42bec7ddca35f5f97c,4205833,released,Primary Tumor,Stage I,...,C3L-01967-DEMO,-23966.0,2018-05-16T14:39:18.135509-05:00,1951.0,b0e2886e-c1ca-4361-bc9b-7aa09b3836df,2024-06-25T17:57:25.646597-05:00,False,NaN,released,NaN
1,e5325dd8-867a-4cc2-8b93-9b0b58fc32f3.rna_seq.a...,16.4316,60.7598,uterus,2029a2c0-7bf4-4fe7-9192-78158d45faa7,5b312edcac677999d84220703070c9f0,4235921,released,Solid Tissue Normal,Stage I,...,C3L-01967-DEMO,-23966.0,2018-05-16T14:39:18.135509-05:00,1951.0,b0e2886e-c1ca-4361-bc9b-7aa09b3836df,2024-06-25T17:57:25.646597-05:00,False,NaN,released,NaN
2,75073929-f7b1-4165-a786-60f733b962d9.rna_seq.a...,87.0767,41.4528,uterus,016ad228-a1b7-4e4e-81f6-c4b45fabb92e,89ee2e7e7be3d6b8eaebb641dd79e594,4243209,released,Primary Tumor,Stage I,...,C3L-01967-DEMO,-23966.0,2018-05-16T14:39:18.135509-05:00,1951.0,b0e2886e-c1ca-4361-bc9b-7aa09b3836df,2024-06-25T17:57:25.646597-05:00,False,NaN,released,NaN
3,119ac668-eab5-42e5-a7a5-f28720173235.rna_seq.a...,74.8179,73.4694,uterus,76aa7262-dd2a-4ff4-8406-8098593911f2,8162ff150f51a2ccd0174989b0b7b379,4232111,released,Primary Tumor,Stage I,...,C3N-01529-DEMO,-25559.0,2018-05-16T14:39:18.135509-05:00,1947.0,25200022-c65d-479b-9b69-2dd914047879,2024-06-25T17:57:25.646597-05:00,False,NaN,released,NaN
4,09c2ef25-0496-4574-995b-d493f9ddf724.rna_seq.a...,102.5040,74.7371,uterus,73f55f88-8817-4ed1-82d7-2da2fe80c308,e45e229d5d828ffc0ce09b925d6601e9,4228697,released,Primary Tumor,Stage I,...,C3N-01007-DEMO,-21777.0,2018-05-16T14:39:18.135509-05:00,1957.0,87929609-cbe0-4a44-8614-fd44c970de57,2024-06-25T17:57:25.646597-05:00,False,NaN,released,NaN


In [29]:
# Specify the cancer type to analyze 
# Analyzed in manuscript ("uterus", "thyroid", "lung", "kidney", "breast", "bone_marrow_and_blood")
cancer = "thyroid"
gene_list = ["HIF1A", "OGT"]

# --- Prerequisites ---
# Before running this script, ensure the following:
# 1. A directory structure exists: './data/TCGA_GeneExpression/{cancer}/'
#    (e.g., './data/TCGA_GeneExpression/uterus/').
# 2. Within the above directory, the following must be present:
#    - A GDC manifest txt file with 'gdc_manifest' in its name. This file must be downloaded
#      from the GDC Data Portal and placed in './data/TCGA_GeneExpression/{cancer}/'.
#    - A subdirectory named 'gene_expression' containing all downloaded gene expression `.tsv` files
#      (also from the GDC Data Portal) for the specified cancer type.  
#      (select cancer type filter > transcriptome profiling > Gene Expression Quantification > RNA-Seq > '.tsv' files)


# --- Step 1: Load the manifest file from GDC ---
# This step loads the manifest file listing metadata about available files for the specified cancer type.
gdc_manifest = load_manifest_file(f'./data/TCGA_GeneExpression/{cancer}/')

# --- Step 2: Fetch clinical data using the GDC API ---
# This step queries the GDC API to fetch clinical data such as diagnosis details and sample types.
# Ensure an active internet connection to access the GDC API.
clinical_manifest = fetch_clinical_data_for_manifest(gdc_manifest, n=None, t=0)

# Save the clinical data for reuse. This file will be saved as 'clinical_gdc_manifest.txt'.
save_clinical_manifest(clinical_manifest, f'./data/TCGA_GeneExpression/{cancer}/')

# --- Step 3: Load and clean the clinical manifest ---
# Load the saved clinical manifest file and clean it by removing irrelevant or empty columns.
clinical_manifest = load_clinical_manifest(f'./data/TCGA_GeneExpression/{cancer}/')
clinical_manifest = clean_dataframe(clinical_manifest)

# --- Step 4: Organize and compile gene expression data ---
# Ensure that the subdirectory './data/TCGA_GeneExpression/{cancer}/gene_expression/' exists
# and contains the downloaded gene expression `.tsv` files from the GDC Data Portal.

# Move all `.tsv` files from subdirectories into the 'gene_expression' directory.
organize_tsv_files(f'./data/TCGA_GeneExpression/{cancer}/gene_expression')

# Load the gene expression data from the `.tsv` files, extracting the specified feature ('fpkm_unstranded').
gene_exps = load_gene_expression_with_filter(
    manifest_df=clinical_manifest,
    exp_dir=f'./data/TCGA_GeneExpression/{cancer}/gene_expression',
    feature='fpkm_unstranded',
    gene_list=gene_list
)

# Save the processed gene expression and gene key data for later use.
save_dataframes(gene_exps, gene_key, cancer)

# --- Step 5: Analyze and verify filenames ---
# Compare filenames in the clinical manifest with those in the 'gene_expression' directory
# to identify any mismatches or missing files.
print(analyze_filenames(clinical_manifest, f'./data/TCGA_GeneExpression/{cancer}/gene_expression'))

# --- Step 6: Reload saved gene expression and gene key data ---
# Reload the saved gene expression and gene key data from the CSV files.
# This is useful for reanalysis or downstream processing without repeating earlier steps.
gene_exps, gene_key = load_dataframes(
    f'./data/TCGA_GeneExpression/{cancer}/gene_expression.csv', 
    f'./data/TCGA_GeneExpression/{cancer}/gene_key.csv'
)

# Create and save the combined DataFrame
clinical_expression_survival_df = create_clinical_expression_survival(
    filtered_gene_exps=gene_exps,
    clinical_manifest=clinical_manifest,
    cancer=cancer,
    save_dir=f'./data/TCGA_GeneExpression/{cancer}/'
)

clinical_expression_survival_df.head()


100%|███████████████████████████████████████| 1402/1402 [23:46<00:00,  1.02s/it]


File saved successfully at ./data/TCGA_GeneExpression/Thyroid/clinical_gdc_manifest.txt
Clinical manifest shape: (1402, 197)
Cleaned clinical manifest shape: (1402, 69)
No directories to clean. Cleaning done.
{'percent_unique_filenames_in_manifest': 100.0, 'percent_overlap_with_directory': 100.0, 'percent_non_overlap_with_directory': 0.0}
Combined DataFrame saved at: ./data/TCGA_GeneExpression/Thyroid/clinical_expression_survival.csv


,filename,HIF1A,OGT,cancer_group,id,md5,size,state,tissue_or_organ_of_origin,age_at_diagnosis,...,demographic_vital_status,demographic_updated_datetime,demographic_submitter_id,demographic_state,demographic_created_datetime,demographic_age_at_index,demographic_days_to_birth,demographic_year_of_birth,demographic_days_to_death,demographic_year_of_death
0,8766ef80-ad3c-4bc0-90fc-c9490a8c136d.rna_seq.a...,28.9954,20.9896,Thyroid,9d3e5c10-4b69-418b-905b-8a9d59f1b046,98744c5f233585065a2892831c2322e9,4257447,released,Thyroid gland,NaN,...,Not Reported,2021-08-06T14:44:01.538025-05:00,REBC-ACBU_demographic,released,2021-03-24T12:18:35.087723-05:00,NaN,NaN,NaN,NaN,NaN
1,6da89125-013a-4cb1-9a0f-220c3a1d35d0.rna_seq.a...,35.3277,44.0968,Thyroid,6bee7d35-0242-4d9a-b342-56f541f9c77c,547c771ff5baa5183a6d4facd21b5812,4278547,released,Thyroid gland,NaN,...,Not Reported,2021-08-06T14:44:01.538025-05:00,REBC-AC9U_demographic,released,2021-03-24T12:18:35.087723-05:00,NaN,NaN,NaN,NaN,NaN
2,d92f9352-85b7-4172-b063-a8145bcfbf42.rna_seq.a...,27.4794,19.6280,Thyroid,1b384a86-e285-486d-abe2-e6e50cb5f8d2,7c5713d1b9673f0cc21adf65c497971d,4228464,released,Thyroid gland,NaN,...,Not Reported,2021-08-06T14:44:01.538025-05:00,REBC-AC9L_demographic,released,2021-03-24T12:18:35.087723-05:00,NaN,NaN,NaN,NaN,NaN
3,9ae05346-a9f7-4326-99d2-d9b4b802e0bd.rna_seq.a...,49.3750,33.3070,Thyroid,5f16ea8f-3d2a-4857-a91c-290259811d6a,91a6e5b6fc2c0dce2985b4cdcc03be02,4250288,released,Thyroid gland,NaN,...,Not Reported,2021-08-06T14:44:01.538025-05:00,REBC-ACBP_demographic,released,2021-03-24T12:18:35.087723-05:00,NaN,NaN,NaN,NaN,NaN
4,05122b8d-865c-4ed4-af10-bbdfa12e12d7.rna_seq.a...,140.5113,31.2582,Thyroid,5c2db08c-9235-4b66-aa23-d131792fa996,2fdae246c29333ec68d82c01d2d4bdac,4248552,released,Thyroid gland,NaN,...,Not Reported,2021-08-06T14:44:01.538025-05:00,REBC-ACBP_demographic,released,2021-03-24T12:18:35.087723-05:00,NaN,NaN,NaN,NaN,NaN


In [30]:
# Specify the cancer type to analyze 
# Analyzed in manuscript ("uterus", "thyroid", "lung", "kidney", "breast", "bone_marrow_and_blood")
cancer = "breast"
gene_list = ["HIF1A", "OGT"]

# --- Prerequisites ---
# Before running this script, ensure the following:
# 1. A directory structure exists: './data/TCGA_GeneExpression/{cancer}/'
#    (e.g., './data/TCGA_GeneExpression/uterus/').
# 2. Within the above directory, the following must be present:
#    - A GDC manifest txt file with 'gdc_manifest' in its name. This file must be downloaded
#      from the GDC Data Portal and placed in './data/TCGA_GeneExpression/{cancer}/'.
#    - A subdirectory named 'gene_expression' containing all downloaded gene expression `.tsv` files
#      (also from the GDC Data Portal) for the specified cancer type.  
#      (select cancer type filter > transcriptome profiling > Gene Expression Quantification > RNA-Seq > '.tsv' files)


# --- Step 1: Load the manifest file from GDC ---
# This step loads the manifest file listing metadata about available files for the specified cancer type.
gdc_manifest = load_manifest_file(f'./data/TCGA_GeneExpression/{cancer}/')

# --- Step 2: Fetch clinical data using the GDC API ---
# This step queries the GDC API to fetch clinical data such as diagnosis details and sample types.
# Ensure an active internet connection to access the GDC API.
clinical_manifest = fetch_clinical_data_for_manifest(gdc_manifest, n=None, t=0)

# Save the clinical data for reuse. This file will be saved as 'clinical_gdc_manifest.txt'.
save_clinical_manifest(clinical_manifest, f'./data/TCGA_GeneExpression/{cancer}/')

# --- Step 3: Load and clean the clinical manifest ---
# Load the saved clinical manifest file and clean it by removing irrelevant or empty columns.
clinical_manifest = load_clinical_manifest(f'./data/TCGA_GeneExpression/{cancer}/')
clinical_manifest = clean_dataframe(clinical_manifest)

# --- Step 4: Organize and compile gene expression data ---
# Ensure that the subdirectory './data/TCGA_GeneExpression/{cancer}/gene_expression/' exists
# and contains the downloaded gene expression `.tsv` files from the GDC Data Portal.

# Move all `.tsv` files from subdirectories into the 'gene_expression' directory.
organize_tsv_files(f'./data/TCGA_GeneExpression/{cancer}/gene_expression')

# Load the gene expression data from the `.tsv` files, extracting the specified feature ('fpkm_unstranded').
gene_exps = load_gene_expression_with_filter(
    manifest_df=clinical_manifest,
    exp_dir=f'./data/TCGA_GeneExpression/{cancer}/gene_expression',
    feature='fpkm_unstranded',
    gene_list=gene_list
)

# Save the processed gene expression and gene key data for later use.
save_dataframes(gene_exps, gene_key, cancer)

# --- Step 5: Analyze and verify filenames ---
# Compare filenames in the clinical manifest with those in the 'gene_expression' directory
# to identify any mismatches or missing files.
print(analyze_filenames(clinical_manifest, f'./data/TCGA_GeneExpression/{cancer}/gene_expression'))

# --- Step 6: Reload saved gene expression and gene key data ---
# Reload the saved gene expression and gene key data from the CSV files.
# This is useful for reanalysis or downstream processing without repeating earlier steps.
gene_exps, gene_key = load_dataframes(
    f'./data/TCGA_GeneExpression/{cancer}/gene_expression.csv', 
    f'./data/TCGA_GeneExpression/{cancer}/gene_key.csv'
)

# Create and save the combined DataFrame
clinical_expression_survival_df = create_clinical_expression_survival(
    filtered_gene_exps=gene_exps,
    clinical_manifest=clinical_manifest,
    cancer=cancer,
    save_dir=f'./data/TCGA_GeneExpression/{cancer}/'
)

clinical_expression_survival_df.head()


100%|███████████████████████████████████████| 1594/1594 [26:51<00:00,  1.01s/it]


File saved successfully at ./data/TCGA_GeneExpression/breast/clinical_gdc_manifest.txt
Clinical manifest shape: (1594, 123)
Cleaned clinical manifest shape: (1594, 55)
No directories to clean. Cleaning done.
{'percent_unique_filenames_in_manifest': 100.0, 'percent_overlap_with_directory': 100.0, 'percent_non_overlap_with_directory': 0.0}
Combined DataFrame saved at: ./data/TCGA_GeneExpression/breast/clinical_expression_survival.csv


,filename,HIF1A,OGT,cancer_group,id,md5,size,state,sample_type,diagnosis_synchronous_malignancy,...,demographic_created_datetime,demographic_year_of_birth,demographic_demographic_id,demographic_updated_datetime,demographic_state,demographic_year_of_death,demographic_days_to_death,diagnosis_diagnosis_is_primary_disease,demographic_age_is_obfuscated,demographic_cause_of_death
0,d0ee5ff7-a49a-4633-93a6-40c9e29fb0b7.rna_seq.a...,30.1536,10.5337,breast,0d140db1-dc0d-431d-b389-846d05bbb827,26025025d0192577ed275c5b45044abc,4243387,released,Primary Tumor,No,...,NaN,1948.0,dc36db44-0161-5ab1-9136-8f3a4f509fb8,2019-07-31T21:38:10.167537-05:00,released,NaN,NaN,NaN,NaN,NaN
1,c58a5583-7b04-4b67-9372-e161e18d7de1.rna_seq.a...,31.4353,16.3492,breast,800ca72f-4bd7-46d7-904b-bcda2fa3500c,b5dc20e532aea0ccd78a17051879c228,4256989,released,Primary Tumor,No,...,NaN,1952.0,887e4eb3-2340-515b-b5b6-7242a16cbd51,2019-07-31T22:06:51.813804-05:00,released,NaN,NaN,NaN,NaN,NaN
2,269c35f0-a4f7-4e30-a69f-f1f3b7b5dace.rna_seq.a...,13.2126,16.9134,breast,958813f4-8036-42f7-856d-7a69c4175adc,0035fb5b56a204c5c0606b98b277344b,4237719,released,Solid Tissue Normal,No,...,NaN,1948.0,dc36db44-0161-5ab1-9136-8f3a4f509fb8,2019-07-31T21:38:10.167537-05:00,released,NaN,NaN,NaN,NaN,NaN
3,158ab1d9-8925-4a05-95da-b2e0ca297474.rna_seq.a...,17.5294,13.7776,breast,c7646bc5-436b-4ad5-b5ae-894b2e843cda,1769f7a4f42471f1f9a8977bae600208,4246730,released,Primary Tumor,No,...,NaN,1952.0,b91af275-dda1-55f0-b973-73bf41c402ce,2019-07-31T21:26:48.925958-05:00,released,NaN,NaN,NaN,NaN,NaN
4,9c2ed2bb-8ee1-441e-9f3b-ffbb4def2673.rna_seq.a...,12.1898,11.1961,breast,90e074a0-ca6d-4031-aa26-f79e5c662ba7,7d8a6169736e3b7b3267fb439690ad40,4244631,released,Primary Tumor,No,...,NaN,1961.0,36849289-31db-5b5f-8cee-b432edd57366,2019-07-31T21:47:13.954244-05:00,released,NaN,NaN,NaN,NaN,NaN


In [31]:
# Specify the cancer type to analyze 
# Analyzed in manuscript ("uterus", "thyroid", "lung", "kidney", "breast", "bone_marrow_and_blood")
cancer = "kidney"
gene_list = ["HIF1A", "OGT"]

# --- Prerequisites ---
# Before running this script, ensure the following:
# 1. A directory structure exists: './data/TCGA_GeneExpression/{cancer}/'
#    (e.g., './data/TCGA_GeneExpression/uterus/').
# 2. Within the above directory, the following must be present:
#    - A GDC manifest txt file with 'gdc_manifest' in its name. This file must be downloaded
#      from the GDC Data Portal and placed in './data/TCGA_GeneExpression/{cancer}/'.
#    - A subdirectory named 'gene_expression' containing all downloaded gene expression `.tsv` files
#      (also from the GDC Data Portal) for the specified cancer type.  
#      (select cancer type filter > transcriptome profiling > Gene Expression Quantification > RNA-Seq > '.tsv' files)


# --- Step 1: Load the manifest file from GDC ---
# This step loads the manifest file listing metadata about available files for the specified cancer type.
gdc_manifest = load_manifest_file(f'./data/TCGA_GeneExpression/{cancer}/')

# --- Step 2: Fetch clinical data using the GDC API ---
# This step queries the GDC API to fetch clinical data such as diagnosis details and sample types.
# Ensure an active internet connection to access the GDC API.
clinical_manifest = fetch_clinical_data_for_manifest(gdc_manifest, n=None, t=0)

# Save the clinical data for reuse. This file will be saved as 'clinical_gdc_manifest.txt'.
save_clinical_manifest(clinical_manifest, f'./data/TCGA_GeneExpression/{cancer}/')

# --- Step 3: Load and clean the clinical manifest ---
# Load the saved clinical manifest file and clean it by removing irrelevant or empty columns.
clinical_manifest = load_clinical_manifest(f'./data/TCGA_GeneExpression/{cancer}/')
clinical_manifest = clean_dataframe(clinical_manifest)

# --- Step 4: Organize and compile gene expression data ---
# Ensure that the subdirectory './data/TCGA_GeneExpression/{cancer}/gene_expression/' exists
# and contains the downloaded gene expression `.tsv` files from the GDC Data Portal.

# Move all `.tsv` files from subdirectories into the 'gene_expression' directory.
organize_tsv_files(f'./data/TCGA_GeneExpression/{cancer}/gene_expression')

# Load the gene expression data from the `.tsv` files, extracting the specified feature ('fpkm_unstranded').
gene_exps = load_gene_expression_with_filter(
    manifest_df=clinical_manifest,
    exp_dir=f'./data/TCGA_GeneExpression/{cancer}/gene_expression',
    feature='fpkm_unstranded',
    gene_list=gene_list
)

# Save the processed gene expression and gene key data for later use.
save_dataframes(gene_exps, gene_key, cancer)

# --- Step 5: Analyze and verify filenames ---
# Compare filenames in the clinical manifest with those in the 'gene_expression' directory
# to identify any mismatches or missing files.
print(analyze_filenames(clinical_manifest, f'./data/TCGA_GeneExpression/{cancer}/gene_expression'))

# --- Step 6: Reload saved gene expression and gene key data ---
# Reload the saved gene expression and gene key data from the CSV files.
# This is useful for reanalysis or downstream processing without repeating earlier steps.
gene_exps, gene_key = load_dataframes(
    f'./data/TCGA_GeneExpression/{cancer}/gene_expression.csv', 
    f'./data/TCGA_GeneExpression/{cancer}/gene_key.csv'
)

# Create and save the combined DataFrame
clinical_expression_survival_df = create_clinical_expression_survival(
    filtered_gene_exps=gene_exps,
    clinical_manifest=clinical_manifest,
    cancer=cancer,
    save_dir=f'./data/TCGA_GeneExpression/{cancer}/'
)

clinical_expression_survival_df.head()


100%|███████████████████████████████████████| 1842/1842 [32:01<00:00,  1.04s/it]


File saved successfully at ./data/TCGA_GeneExpression/kidney/clinical_gdc_manifest.txt
Clinical manifest shape: (1842, 125)
Cleaned clinical manifest shape: (1842, 69)
No directories to clean. Cleaning done.
{'percent_unique_filenames_in_manifest': 100.0, 'percent_overlap_with_directory': 100.0, 'percent_non_overlap_with_directory': 0.0}
Combined DataFrame saved at: ./data/TCGA_GeneExpression/kidney/clinical_expression_survival.csv


,filename,HIF1A,OGT,cancer_group,id,md5,size,state,sample_type,diagnosis_ajcc_pathologic_stage,...,demographic_days_to_birth,demographic_created_datetime,demographic_year_of_birth,demographic_demographic_id,demographic_updated_datetime,demographic_age_is_obfuscated,demographic_days_to_death,demographic_state,demographic_year_of_death,diagnosis_pediatric_kidney_staging
0,1a1c36d6-99b6-4e57-84ac-c2759d87e543.rna_seq.a...,137.0645,24.8498,kidney,334ed773-1389-4c98-ad1b-b87a54743ad2,da719b3284b890eec8fcd64eacb5e2a4,4229525,released,Next Generation Cancer Model,NaN,...,-941.0,2019-04-04T13:42:46.632048-05:00,2012.0,70c3ac33-e2b8-4a33-8214-e451a903b927,2021-07-12T12:25:55.528644-05:00,False,NaN,released,NaN,NaN
1,9e99b6c3-512d-4948-bfed-1ab6ba447a8f.rna_seq.a...,100.6179,16.9538,kidney,4d0ca4d7-dc19-4d37-8787-2884fe5f6d05,5e6b834b40382bd1c6c25bdbb08f4ab3,4209631,released,Expanded Next Generation Cancer Model,NaN,...,-941.0,2019-04-04T13:42:46.632048-05:00,2012.0,70c3ac33-e2b8-4a33-8214-e451a903b927,2021-07-12T12:25:55.528644-05:00,False,NaN,released,NaN,NaN
2,bdef2d26-c8a9-4675-90d2-78ac46b4369c.rna_seq.a...,27.2636,43.1209,kidney,9b00d458-e1a2-42d8-bc98-d7e8b9c91987,3ee2c983ebb18a3d7701c9430f40b0a3,4228647,released,Primary Tumor,Stage I,...,-24933.0,2021-01-14T15:58:04.601672-06:00,1948.0,f4254aa9-691d-4d03-9421-3c1f0e24e24e,2024-06-25T17:57:25.646597-05:00,False,NaN,released,NaN,NaN
3,87eca657-e2ec-433a-b233-4966c8e8920a.rna_seq.a...,51.9599,53.1540,kidney,3c02062d-e0e7-4287-a48c-704c832f8742,0ff31e30726ce34a22cb4a50925c70bf,4247753,released,Solid Tissue Normal,Stage I,...,-25175.0,2021-01-14T15:58:04.601672-06:00,1948.0,e1f9bcd2-7746-44c7-9e98-c4eebc0e2b95,2024-06-25T17:57:25.646597-05:00,False,1224.0,released,2021.0,NaN
4,55f58be6-fa90-4898-ab19-00687a1cdc58.rna_seq.a...,40.5800,38.9887,kidney,b0bad1c7-9f79-470f-ac66-df0daff66cf7,9443ab44cbcf2621e58b805dc8da9be2,4227286,released,Primary Tumor,Stage I,...,-24969.0,2021-01-14T15:58:04.601672-06:00,1948.0,115e1cfc-c60e-4f3d-b4d8-c9e26eb3613e,2024-06-25T17:57:25.646597-05:00,False,NaN,released,NaN,NaN


In [33]:
# Specify the cancer type to analyze 
# Analyzed in manuscript ("uterus", "thyroid", "lung", "kidney", "breast", "bone_marrow_and_blood")
cancer = "lung"
gene_list = ["HIF1A", "OGT"]

# --- Prerequisites ---
# Before running this script, ensure the following:
# 1. A directory structure exists: './data/TCGA_GeneExpression/{cancer}/'
#    (e.g., './data/TCGA_GeneExpression/uterus/').
# 2. Within the above directory, the following must be present:
#    - A GDC manifest txt file with 'gdc_manifest' in its name. This file must be downloaded
#      from the GDC Data Portal and placed in './data/TCGA_GeneExpression/{cancer}/'.
#    - A subdirectory named 'gene_expression' containing all downloaded gene expression `.tsv` files
#      (also from the GDC Data Portal) for the specified cancer type.  
#      (select cancer type filter > transcriptome profiling > Gene Expression Quantification > RNA-Seq > '.tsv' files)


# --- Step 1: Load the manifest file from GDC ---
# This step loads the manifest file listing metadata about available files for the specified cancer type.
gdc_manifest = load_manifest_file(f'./data/TCGA_GeneExpression/{cancer}/')

# --- Step 2: Fetch clinical data using the GDC API ---
# This step queries the GDC API to fetch clinical data such as diagnosis details and sample types.
# Ensure an active internet connection to access the GDC API.
clinical_manifest = fetch_clinical_data_for_manifest(gdc_manifest, n=None, t=0)

# Save the clinical data for reuse. This file will be saved as 'clinical_gdc_manifest.txt'.
save_clinical_manifest(clinical_manifest, f'./data/TCGA_GeneExpression/{cancer}/')

# --- Step 3: Load and clean the clinical manifest ---
# Load the saved clinical manifest file and clean it by removing irrelevant or empty columns.
clinical_manifest = load_clinical_manifest(f'./data/TCGA_GeneExpression/{cancer}/')
clinical_manifest = clean_dataframe(clinical_manifest)

# --- Step 4: Organize and compile gene expression data ---
# Ensure that the subdirectory './data/TCGA_GeneExpression/{cancer}/gene_expression/' exists
# and contains the downloaded gene expression `.tsv` files from the GDC Data Portal.

# Move all `.tsv` files from subdirectories into the 'gene_expression' directory.
organize_tsv_files(f'./data/TCGA_GeneExpression/{cancer}/gene_expression')

# Load the gene expression data from the `.tsv` files, extracting the specified feature ('fpkm_unstranded').
gene_exps = load_gene_expression_with_filter(
    manifest_df=clinical_manifest,
    exp_dir=f'./data/TCGA_GeneExpression/{cancer}/gene_expression',
    feature='fpkm_unstranded',
    gene_list=gene_list
)

# Save the processed gene expression and gene key data for later use.
save_dataframes(gene_exps, gene_key, cancer)

# --- Step 5: Analyze and verify filenames ---
# Compare filenames in the clinical manifest with those in the 'gene_expression' directory
# to identify any mismatches or missing files.
print(analyze_filenames(clinical_manifest, f'./data/TCGA_GeneExpression/{cancer}/gene_expression'))

# --- Step 6: Reload saved gene expression and gene key data ---
# Reload the saved gene expression and gene key data from the CSV files.
# This is useful for reanalysis or downstream processing without repeating earlier steps.
gene_exps, gene_key = load_dataframes(
    f'./data/TCGA_GeneExpression/{cancer}/gene_expression.csv', 
    f'./data/TCGA_GeneExpression/{cancer}/gene_key.csv'
)

# Create and save the combined DataFrame
clinical_expression_survival_df = create_clinical_expression_survival(
    filtered_gene_exps=gene_exps,
    clinical_manifest=clinical_manifest,
    cancer=cancer,
    save_dir=f'./data/TCGA_GeneExpression/{cancer}/'
)

clinical_expression_survival_df.head()

100%|███████████████████████████████████████| 1996/1996 [51:30<00:00,  1.55s/it]


File saved successfully at ./data/TCGA_GeneExpression/lung/clinical_gdc_manifest.txt
Clinical manifest shape: (1996, 124)
Cleaned clinical manifest shape: (1996, 66)
No directories to clean. Cleaning done.
{'percent_unique_filenames_in_manifest': 100.0, 'percent_overlap_with_directory': 100.0, 'percent_non_overlap_with_directory': 0.0}
Combined DataFrame saved at: ./data/TCGA_GeneExpression/lung/clinical_expression_survival.csv


,filename,HIF1A,OGT,cancer_group,id,md5,size,state,sample_type,diagnosis_ajcc_pathologic_stage,...,demographic_submitter_id,demographic_days_to_birth,demographic_created_datetime,demographic_year_of_birth,demographic_demographic_id,demographic_updated_datetime,demographic_age_is_obfuscated,demographic_days_to_death,demographic_state,demographic_year_of_death
0,c42bc86c-11ae-4b1b-b281-dfdd2e2250a6.rna_seq.a...,39.2168,18.5124,lung,4a5cb3ce-9784-4d1f-8fae-50602d121aba,e57aa56bc3bbd01efa87f13c3f707c6a,4230270,released,Metastatic,Stage IV,...,HCM-BROD-0027-C34_demographic,-23897.0,2019-04-04T15:11:28.695214-05:00,1950.0,7dc95354-684a-4dd2-9ea8-f32985ba4049,2020-10-08T15:36:07.819253-05:00,False,NaN,released,NaN
1,1cb15aef-b8b2-4cbd-bd99-2e0c6df9d6e1.rna_seq.a...,70.3471,16.3804,lung,a0fbb2d1-014e-4021-ad8a-a0a16ce47789,5453b68ba3b5f515642b3ef745a230d1,4212122,released,Next Generation Cancer Model,Stage IV,...,HCM-BROD-0027-C34_demographic,-23897.0,2019-04-04T15:11:28.695214-05:00,1950.0,7dc95354-684a-4dd2-9ea8-f32985ba4049,2020-10-08T15:36:07.819253-05:00,False,NaN,released,NaN
2,eb660889-0075-4f0d-ae2e-868616f02be3.rna_seq.a...,28.2381,44.2673,lung,2984dbfd-3385-441d-9d4f-58cbc6093887,43ec414f605cca3b8ae49dd2472467d8,4245476,released,Solid Tissue Normal,Stage IIB,...,C3N-00497-DEMO,-23739.0,2017-11-01T09:34:04.825788-05:00,1951.0,0f168cdd-97d5-44a9-a5f9-ab9ecba0db2f,2024-06-25T17:57:25.646597-05:00,False,NaN,released,NaN
3,1c4134e9-1bd6-455f-afa7-98a5d0d501c4.rna_seq.a...,60.6856,30.9914,lung,0791f456-b9b0-456d-ae7b-52534bfb485f,748be67cf9a07da6a2c1ddb09787e938,4226619,released,Primary Tumor,Stage IIIA,...,C3N-01408-DEMO,-21281.0,2018-05-16T14:39:18.135509-05:00,1959.0,169c074b-f0c9-4538-b310-e53a926e25a2,2024-06-25T17:57:25.646597-05:00,False,NaN,released,NaN
4,557c4895-d5f3-4005-abcd-84dafb75f992.rna_seq.a...,28.5966,48.0258,lung,2f38a7d6-b3a8-4877-996c-039117fd5a41,d423a5b6282756cb4c0584d808a0a9ba,4234462,released,Solid Tissue Normal,Stage IIIA,...,C3N-01408-DEMO,-21281.0,2018-05-16T14:39:18.135509-05:00,1959.0,169c074b-f0c9-4538-b310-e53a926e25a2,2024-06-25T17:57:25.646597-05:00,False,NaN,released,NaN


In [34]:
# Specify the cancer type to analyze 
# Analyzed in manuscript ("uterus", "thyroid", "lung", "kidney", "breast", "bone_marrow_and_blood")
cancer = "bone_marrow_and_blood"
gene_list = ["HIF1A", "OGT"]

# --- Prerequisites ---
# Before running this script, ensure the following:
# 1. A directory structure exists: './data/TCGA_GeneExpression/{cancer}/'
#    (e.g., './data/TCGA_GeneExpression/uterus/').
# 2. Within the above directory, the following must be present:
#    - A GDC manifest txt file with 'gdc_manifest' in its name. This file must be downloaded
#      from the GDC Data Portal and placed in './data/TCGA_GeneExpression/{cancer}/'.
#    - A subdirectory named 'gene_expression' containing all downloaded gene expression `.tsv` files
#      (also from the GDC Data Portal) for the specified cancer type.  
#      (select cancer type filter > transcriptome profiling > Gene Expression Quantification > RNA-Seq > '.tsv' files)


# --- Step 1: Load the manifest file from GDC ---
# This step loads the manifest file listing metadata about available files for the specified cancer type.
gdc_manifest = load_manifest_file(f'./data/TCGA_GeneExpression/{cancer}/')

# --- Step 2: Fetch clinical data using the GDC API ---
# This step queries the GDC API to fetch clinical data such as diagnosis details and sample types.
# Ensure an active internet connection to access the GDC API.
clinical_manifest = fetch_clinical_data_for_manifest(gdc_manifest, n=None, t=0)

# Save the clinical data for reuse. This file will be saved as 'clinical_gdc_manifest.txt'.
save_clinical_manifest(clinical_manifest, f'./data/TCGA_GeneExpression/{cancer}/')

# --- Step 3: Load and clean the clinical manifest ---
# Load the saved clinical manifest file and clean it by removing irrelevant or empty columns.
clinical_manifest = load_clinical_manifest(f'./data/TCGA_GeneExpression/{cancer}/')
clinical_manifest = clean_dataframe(clinical_manifest)

# --- Step 4: Organize and compile gene expression data ---
# Ensure that the subdirectory './data/TCGA_GeneExpression/{cancer}/gene_expression/' exists
# and contains the downloaded gene expression `.tsv` files from the GDC Data Portal.

# Move all `.tsv` files from subdirectories into the 'gene_expression' directory.
organize_tsv_files(f'./data/TCGA_GeneExpression/{cancer}/gene_expression')

# Load the gene expression data from the `.tsv` files, extracting the specified feature ('fpkm_unstranded').
gene_exps = load_gene_expression_with_filter(
    manifest_df=clinical_manifest,
    exp_dir=f'./data/TCGA_GeneExpression/{cancer}/gene_expression',
    feature='fpkm_unstranded',
    gene_list=gene_list
)

# Save the processed gene expression and gene key data for later use.
save_dataframes(gene_exps, gene_key, cancer)

# --- Step 5: Analyze and verify filenames ---
# Compare filenames in the clinical manifest with those in the 'gene_expression' directory
# to identify any mismatches or missing files.
print(analyze_filenames(clinical_manifest, f'./data/TCGA_GeneExpression/{cancer}/gene_expression'))

# --- Step 6: Reload saved gene expression and gene key data ---
# Reload the saved gene expression and gene key data from the CSV files.
# This is useful for reanalysis or downstream processing without repeating earlier steps.
gene_exps, gene_key = load_dataframes(
    f'./data/TCGA_GeneExpression/{cancer}/gene_expression.csv', 
    f'./data/TCGA_GeneExpression/{cancer}/gene_key.csv'
)

# Create and save the combined DataFrame
clinical_expression_survival_df = create_clinical_expression_survival(
    filtered_gene_exps=gene_exps,
    clinical_manifest=clinical_manifest,
    cancer=cancer,
    save_dir=f'./data/TCGA_GeneExpression/{cancer}/'
)

clinical_expression_survival_df.head()

 23%|████████▌                            | 1661/7224 [36:26<2:26:10,  1.58s/it]

Failed to retrieve sample type for UUID e2c2ab79-efe3-4c71-8a50-fbd2dec779f5.
Failed to retrieve diagnosis details for UUID e2c2ab79-efe3-4c71-8a50-fbd2dec779f5.


 23%|████████▌                            | 1662/7224 [36:29<3:23:20,  2.19s/it]

Failed to retrieve demographic details for UUID e2c2ab79-efe3-4c71-8a50-fbd2dec779f5.


 25%|█████████▎                           | 1824/7224 [39:54<2:02:34,  1.36s/it]

...

 61%|█████████████████████▎             | 4391/7224 [1:40:35<1:08:39,  1.45s/it]

Failed to retrieve demographic details for UUID e0170e3b-4ded-41b8-a641-4b52ba358984.


100%|█████████████████████████████████████| 7224/7224 [3:18:35<00:00,  1.65s/it]


File saved successfully at ./data/TCGA_GeneExpression/bone_marrow_and_blood/clinical_gdc_manifest.txt
Clinical manifest shape: (7224, 125)
Cleaned clinical manifest shape: (7224, 60)
No directories to clean. Cleaning done.
{'percent_unique_filenames_in_manifest': 100.0, 'percent_overlap_with_directory': 100.0, 'percent_non_overlap_with_directory': 0.0}
Combined DataFrame saved at: ./data/TCGA_GeneExpression/bone_marrow_and_blood/clinical_expression_survival.csv


,filename,HIF1A,OGT,cancer_group,id,md5,size,state,sample_type,diagnosis_synchronous_malignancy,...,demographic_age_at_index,demographic_submitter_id,demographic_state,demographic_days_to_birth,demographic_created_datetime,demographic_cause_of_death,demographic_days_to_death,demographic_year_of_birth,demographic_year_of_death,demographic_country_of_residence_at_enrollment
0,3c1240da-a695-46d6-9de1-a89ddd5f7ca8.rna_seq.a...,30.2218,96.0223,bone_marrow_and_blood,36b2de16-e1f8-4788-a972-9ea8b8fd9ad6,b879a6a553a1cfc5b5c393effeb6701b,4176618,released,FFPE Scrolls,No,...,28.0,BLGSP-71-30-00672_demographic,released,-10451.0,2021-02-12T14:31:57.175137-06:00,NaN,NaN,NaN,NaN,NaN
1,1345f272-89e8-4c1a-865f-95059c8c1816.rna_seq.a...,34.5952,133.1021,bone_marrow_and_blood,9d6d0ffe-421f-4724-9e80-dacf7255e625,f20dba39bb45129cd8eac47d665acd79,4226371,released,FFPE Scrolls,No,...,61.0,BLGSP-71-30-00651_demographic,released,-22297.0,2021-02-12T14:16:25.524723-06:00,Cancer Related,228.0,NaN,NaN,NaN
2,7b733f14-fac0-4617-a332-4cdec12412aa.rna_seq.a...,23.1050,70.3996,bone_marrow_and_blood,d9e81d4a-0f7f-4105-aec6-0e51cbf5b94f,c1f80b4df4a6e5e29a1ef1fdc9f346de,4241380,released,FFPE Scrolls,No,...,66.0,BLGSP-71-30-00660_demographic,released,-24272.0,2021-02-12T14:21:33.510993-06:00,NaN,NaN,NaN,NaN,NaN
3,f02be941-5cca-4b9a-b975-6ba11225cdd9.rna_seq.a...,56.2326,84.5481,bone_marrow_and_blood,bfb55897-7a02-4bd3-abb4-ba750a3889c0,9a1dc196dc14fbe5a6dce5d1235b8eeb,4231627,released,FFPE Scrolls,No,...,67.0,BLGSP-71-30-00647_demographic,released,-24516.0,2021-02-12T14:13:20.746038-06:00,Cancer Related,1224.0,NaN,NaN,NaN
4,a671163b-2faf-49ae-b058-99bd3aa7787a.rna_seq.a...,59.7714,82.9424,bone_marrow_and_blood,003c1a27-24fc-41f3-8de8-9ce029c60adb,f8b6c1ad840516260457bc68d882dcfa,4224063,released,Primary Tumor,No,...,7.0,BLGSP-71-08-00059_demographic,released,-2684.0,2018-05-30T12:35:35.862881-05:00,NaN,NaN,NaN,NaN,NaN
